In [2]:
import os
from langchain_community.document_loaders import  PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [3]:


### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                # keep add the other metadata if needed

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/")


Found 1 PDF files to process

Processing: IEC-60870-5-104-2006.pdf
  ✓ Loaded 14 pages

Total documents loaded: 14


### Fixed Size Chunking

#### Simple but inefficient for longer text

In [4]:

def fixed_size_chunks(text, chunk_size):
    """Split text into fixed-size chunks"""
    words = text.split()
    chunks = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]
    return [' '.join(chunk) for chunk in chunks]

all_pdf_documents = process_all_pdfs("../data/")
text = " ".join([doc.page_content for doc in all_pdf_documents])
chunks = fixed_size_chunks(text, chunk_size=100)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")

Found 1 PDF files to process

Processing: IEC-60870-5-104-2006.pdf
  ✓ Loaded 14 pages

Total documents loaded: 14
Chunk 1:
Telecontrol equipment and systems – Part 5-104: Transmission protocols – Network access for IEC 60870-5-101 using standard transport profiles Reference number IEC 60870-5-104:2006(E) INTERNATIONAL STANDARD IEC 60870-5-104 Second edition 2006-06 This English-language version is derived from the original bilingual publication by leaving out all French-language pages. Missing page numbers correspond to the French- language pages. iTeh Standards (https://standards.iteh.ai) Document Preview IEC 60870-5-104:2006 https://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006 Publication numbering As from 1 January 1997 all IEC publications are issued with a designation in the 60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. Consolidated editions The IEC is now publishing

Chunk 2:
consolidated versions of i

### Sentence Base Chunking

In [5]:
from nltk.tokenize import sent_tokenize
import nltk

nltk.download('punkt')

text = " ".join([doc.page_content for doc in all_pdf_documents])
sentences = sent_tokenize(text) 

for i, sentence in enumerate(sentences):
    print(f"Sentence {i + 1}:\n{sentence}\n")



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rohit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - 'C:\\Users\\Rohit/nltk_data'
    - 'c:\\RAG_POC\\venv\\nltk_data'
    - 'c:\\RAG_POC\\venv\\share\\nltk_data'
    - 'c:\\RAG_POC\\venv\\lib\\nltk_data'
    - 'C:\\Users\\Rohit\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


### Document Base Chunking

### Semantic-Baseed Chuncking 

Lable Base chunking 

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np

sentences = [doc.page_content for doc in all_pdf_documents]


model = SentenceTransformer('all-mpnet-base-v2')

# Create embeddings for each sentence
embeddings = model.encode(sentences)


# Use KMeans to cluster the embeddings for each sentence

KMeans= KMeans(n_clusters=5, random_state=0)
lables = KMeans.fit_predict(embeddings) 

# Group sentences by their cluster labels
clustered_sentences = {}
for i, label in enumerate(lables):
    if label not in clustered_sentences:
        clustered_sentences[label] = []
    clustered_sentences[label].append(sentences[i])



#  Print the semantically similar sentences in each cluster
for label, sentence_list in clustered_sentences.items():
    print(f"Cluster {label + 1}:")
    for sentence in sentence_list:
        print(f"  - {sentence}")
    print("\n")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6166.64it/s]


Cluster 1:
  - Telecontrol equipment and systems – 
Part 5-104: 
Transmission protocols – 
Network access for IEC 60870-5-101  
using standard transport profiles 
 
 
 
Reference number 
IEC 60870-5-104:2006(E) 
INTERNATIONAL 
STANDARD 
IEC
60870-5-104
 
Second edition
2006-06
 
This English-language version is derived from the original 
bilingual publication by leaving out all French-language 
pages. Missing page numbers correspond to the French-
language pages. 
iTeh Standards
(https://standards.iteh.ai)
Document Preview
IEC 60870-5-104:2006
https://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006
  - Publication numbering 
As from 1 January 1997 all IEC publications are issued with a designation in the 
60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. 
Consolidated editions 
The IEC is now publishing consolidated versions of its publications. For example, 
edition numbers 1.0, 1.1 and 1.2 refer, respectively, to 

In [ ]:
sentences = [doc.page_content for doc in all_pdf_documents]
sentences

# text = " ".join([doc.page_content for doc in all_pdf_documents])
# text

['Telecontrol equipment and systems – \nPart 5-104: \nTransmission protocols – \nNetwork access for IEC 60870-5-101  \nusing standard transport profiles \n \n \n \nReference number \nIEC 60870-5-104:2006(E) \nINTERNATIONAL \nSTANDARD \nIEC\n60870-5-104\n \nSecond edition\n2006-06\n \nThis English-language version is derived from the original \nbilingual publication by leaving out all French-language \npages. Missing page numbers correspond to the French-\nlanguage pages. \niTeh Standards\n(https://standards.iteh.ai)\nDocument Preview\nIEC 60870-5-104:2006\nhttps://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006',
 'Publication numbering \nAs from 1 January 1997 all IEC publications are issued with a designation in the \n60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. \nConsolidated editions \nThe IEC is now publishing consolidated versions of its publications. For example, \nedition numbers 1.0, 1.1 and 1.2 refer,

### Overlapping Chunking 
UseCase in machine translation, summarization, questing-Answering where the meaning of adjacent chunks is highly dependent on each other

In [ ]:

def Overlapping_chunks(text, chunk_size, overlap):
    """Split text into overlapping chunks"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks


text = " ".join([doc.page_content for doc in all_pdf_documents])
chunks = Overlapping_chunks(text, chunk_size=100, overlap=20)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")


Chunk 1:
Telecontrol equipment and systems – Part 5-104: Transmission protocols – Network access for IEC 60870-5-101 using standard transport profiles Reference number IEC 60870-5-104:2006(E) INTERNATIONAL STANDARD IEC 60870-5-104 Second edition 2006-06 This English-language version is derived from the original bilingual publication by leaving out all French-language pages. Missing page numbers correspond to the French- language pages. iTeh Standards (https://standards.iteh.ai) Document Preview IEC 60870-5-104:2006 https://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006 Publication numbering As from 1 January 1997 all IEC publications are issued with a designation in the 60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. Consolidated editions The IEC is now publishing

Chunk 2:
60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. Consolidated editions The IEC is now publishing consolidated versions 

### Recursive Chunking 

Work as Hierarchical and iterative manner, structure based on content boundaries

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = " ".join([doc.page_content for doc in all_pdf_documents])

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=20
)
chunks = text_splitter.split_text(text) 

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")



Chunk 1:
Telecontrol equipment and systems – 
Part 5-104: 
Transmission protocols – 
Network access for IEC 60870-5-101  
using standard transport profiles 
 
 
 
Reference number 
IEC 60870-5-104:2006(E) 
INTERNATIONAL 
STANDARD 
IEC
60870-5-104
 
Second edition
2006-06
 
This English-language version is derived from the original 
bilingual publication by leaving out all French-language 
pages. Missing page numbers correspond to the French-
language pages. 
iTeh Standards
(https://standards.iteh.ai)

Chunk 2:
Document Preview
IEC 60870-5-104:2006
https://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006 Publication numbering 
As from 1 January 1997 all IEC publications are issued with a designation in the 
60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. 
Consolidated editions 
The IEC is now publishing consolidated versions of its publications. For example,

Chunk 3:
edition numbers 1.0, 1.1 and 1.2 refer, respecti

### Agentic chunking 
LLM base chunking , Create chunks based on relevant infomation.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama
from langchain.chains import LLMChain

# Initialize your LLM
llm = Ollama(model="llama3.1:1.8b-instruct-q4_K_M")

# Define a prompt that asks the LLM to chunk text semantically
chunk_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an intelligent document chunker.
Split the following text into coherent chunks of 200-300 words,
ensuring each chunk represents a complete idea or section.

Text:
{text}

Return the chunks as a numbered list.
    """
)

chunk_chain = LLMChain(llm=llm, prompt=chunk_prompt)

# Example usage
doc_text = "Your long PDF-extracted text here..."
chunks = chunk_chain.run(doc_text)

print(chunks)

ModuleNotFoundError: No module named 'langchain.prompts'

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

def chunk_text(text, max_length=512, stride=256):
    """Chunk text into overlapping segments based on token length."""
    input_ids = tokenizer(text, return_tensors="pt").input_ids[0]

    chunks = []
    i=0
    while i < len(input_ids):
        end = min(i + max_length, len(input_ids))
        chunk_ids = input_ids[i:end]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_text)
        i += stride  # Move forward by the stride length
        return chunks
    
    text = " ".join([doc.page_content for doc in all_pdf_documents])
    chunks = chunk_text(text, max_length=512, stride=256)   

    for chunk in chunks:
        print(f"Chunk:\n{chunk}\n")

c:\RAG_POC\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohit\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 10887.91it/s]


### Token Base  chunking
 Fixed number of token rather than sentense and word / not aleways allign with natural language  =boundary, works with LLM

In [10]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def chunk_text_with_gpt2(text, max_length=512):
    tokens = tokenizer.encode(text)
    chunks = [tokens[i:i + max_length] for i in range(0, len(tokens), max_length)]
    return [tokenizer.decode(chunk) for chunk in chunks]

text = " ".join([doc.page_content for doc in all_pdf_documents])
token_chunks = chunk_text_with_gpt2(text, 512)

for i, chunk in enumerate(token_chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8140 > 1024). Running this sequence through the model will result in indexing errors


Chunk 1:
Telecontrol equipment and systems – 
Part 5-104: 
Transmission protocols – 
Network access for IEC 60870-5-101  
using standard transport profiles 
 
 
 
Reference number 
IEC 60870-5-104:2006(E) 
INTERNATIONAL 
STANDARD 
IEC
60870-5-104
 
Second edition
2006-06
 
This English-language version is derived from the original 
bilingual publication by leaving out all French-language 
pages. Missing page numbers correspond to the French-
language pages. 
iTeh Standards
(https://standards.iteh.ai)
Document Preview
IEC 60870-5-104:2006
https://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006 Publication numbering 
As from 1 January 1997 all IEC publications are issued with a designation in the 
60000 series. For example, IEC 34-1 is now referred to as IEC 60034-1. 
Consolidated editions 
The IEC is now publishing consolidated versions of its publications. For example, 
edition numbers 1.0, 1.1 and 1.2 refer, respectively, to the base p

### Topic Base chunking 
use in articles, research papers, or reports with diverse subject matter

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np


def topic_based_chunks(text, num_topics=5):

    sentences = text.split('. ')

    vectorizer = CountVectorizer()
    sentence_vectors = vectorizer.fit_transform(sentences)

    lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda.fit_transform(sentence_vectors)

    # GET the topic distribution for each sentence
    topic_word = lda.components_
    vocab = vectorizer.get_feature_names_out()

    # Identify the top words for each topic
    topic = []
    for topic_idx, topic_dist in enumerate(topic_word):
        top_words_idx = topic_dist.argsort()[-10:][::-1]
        top_words = [vocab[i] for i in top_words_idx]
        topic.append("Topic {}: {}".format(topic_idx, ", ".join(top_words)))

    # Generate chunks based on topic distribution
    chunk_with_topic=[] 
    for i, sentence in enumerate(sentences):
        topic_distribution = lda.transform(vectorizer.transform([sentence]))
        dominant_topic = np.argmax(topic_distribution)
        chunk_with_topic.append((topic[dominant_topic], sentence)) 


    return chunk_with_topic


text = " ".join([doc.page_content for doc in all_pdf_documents])

chunks = topic_based_chunks(text, num_topics=5)




In [13]:
text = " ".join([doc.page_content for doc in all_pdf_documents])

chunks = topic_based_chunks(text, num_topics=5)
chunks

[('Topic 2: the, iec, of, for, part, transmission, in, and, protocols, this',
  'Telecontrol equipment and systems – \nPart 5-104: \nTransmission protocols – \nNetwork access for IEC 60870-5-101  \nusing standard transport profiles \n \n \n \nReference number \nIEC 60870-5-104:2006(E) \nINTERNATIONAL \nSTANDARD \nIEC\n60870-5-104\n \nSecond edition\n2006-06\n \nThis English-language version is derived from the original \nbilingual publication by leaving out all French-language \npages'),
 ('Topic 0: isdn, 25, fr, router, ip, tcp, the, network, 115, general',
  'Missing page numbers correspond to the French-\nlanguage pages'),
 ('Topic 4: iec, 60870, standards, of, 104, iteh, 2006, figure, ai, https',
  '\niTeh Standards\n(https://standards.iteh.ai)\nDocument Preview\nIEC 60870-5-104:2006\nhttps://standards.iteh.ai/catalog/standards/iec/a0f6f64b-a9da-4964-b973-00b7ec19106a/iec-60870-5-104-2006 Publication numbering \nAs from 1 January 1997 all IEC publications are issued with a designat

### Keyword base chunking 

Capture the natural topic break based on keyword

In [15]:
def keyword_based_chunks(text, keywords):
    """Split text into chunks based on the presence of specific keywords."""
    sentences = text.split('. ')
    chunks = []
    current_chunk = []

    for sentence in sentences:
        current_chunk.append(sentence)
        if any(keyword in sentence for keyword in keywords):
            if current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentence]
        else:
            current_chunk.append(sentence)

    if current_chunk:
        chunks.append('\n'.join(current_chunk))
    return chunks

text = " ".join([doc.page_content for doc in all_pdf_documents])
keywords = ["Introduction", "Conclusion", "Summary", "Methodology", "Results"]
keyword_chunks = keyword_based_chunks(text, keywords)
for i, chunk in enumerate(keyword_chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")

             

Chunk 1:
Telecontrol equipment and systems – 
Part 5-104: 
Transmission protocols – 
Network access for IEC 60870-5-101  
using standard transport profiles 
 
 
 
Reference number 
IEC 60870-5-104:2006(E) 
INTERNATIONAL 
STANDARD 
IEC
60870-5-104
 
Second edition
2006-06
 
This English-language version is derived from the original 
bilingual publication by leaving out all French-language 
pages
Telecontrol equipment and systems – 
Part 5-104: 
Transmission protocols – 
Network access for IEC 60870-5-101  
using standard transport profiles 
 
 
 
Reference number 
IEC 60870-5-104:2006(E) 
INTERNATIONAL 
STANDARD 
IEC
60870-5-104
 
Second edition
2006-06
 
This English-language version is derived from the original 
bilingual publication by leaving out all French-language 
pages
Missing page numbers correspond to the French-
language pages
Missing page numbers correspond to the French-
language pages

iTeh Standards
(https://standards.iteh.ai)
Document Preview
IEC 60870-5-104:2006
https:/